# Ideal gas equation of state using grand canonical ensemble transition-matrix Monte Carlo

In this example, the ideal gas equation of state is obtained as a test of the flat histogram method.

In [1]:
params={"cubic_side_length": 8, "beta": 1./1.2, "mu": -3}
script="""
MonteCarlo
Configuration cubic_side_length={cubic_side_length} particle_type=atom:/feasst/particle/atom.txt
Potential Model=IdealGas
ThermoParams beta={beta} chemical_potential={mu}
FlatHistogram Macrostate=MacrostateNumParticles width=1 min=0 max=50 \
              Bias=TransitionMatrix min_sweeps=100
TrialTransfer particle_type=atom
CriteriaUpdater trials_per_update=1e5
CriteriaWriter trials_per_write=1e5 output_file=id_fh.txt
Run until=complete
""".format(**params)

def run(script):
    with open('script0.txt', 'w') as file: file.write(script)
    import subprocess
    syscode = subprocess.call("feasst < script0.txt > script0.log", shell=True, executable='/bin/bash')
    with open('script0.log', 'r') as file: print(file.read(), '\n', 'exit:', syscode)
run(script)

# Usage: feasst < file.txt
# For more information, use the command "feasst-menu"
# Exit with ctrl-c
FEASST version 0.25.19
MonteCarlo
Configuration cubic_side_length=8 particle_type=atom:/feasst/particle/atom.txt
Potential Model=IdealGas
ThermoParams beta=0.8333333333333334 chemical_potential=-3
FlatHistogram Bias=TransitionMatrix Macrostate=MacrostateNumParticles max=50 min=0 min_sweeps=100 width=1
TrialTransfer particle_type=atom
CriteriaUpdater trials_per_update=1e5
CriteriaWriter output_file=id_fh.txt trials_per_write=1e5
Run until=complete
# Initializing random number generator with seed: 1781198252
 
 exit: 0


Check the ideal gas relationship, $\beta P = \rho$

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

fh=pd.read_csv('id_fh.txt', comment="#")
#print(fh)
print('N betaPV difference')
for delta_conjugate in np.arange(-5, 5, 0.1):
    fh['ln_prob_rw'] = fh['ln_prob'] + fh['state']*delta_conjugate - fh['ln_prob'].min()  # avoid negative log
    fh['ln_prob_rw'] -= np.log(sum(np.exp(fh['ln_prob_rw'])))   # renormalize
    if fh['ln_prob_rw'].values[-1] < -6:
        #plt.plot(fh['ln_prob_rw'])
        N = (np.exp(fh["ln_prob_rw"]) * fh["state"]).sum()
        betaPV = -fh["ln_prob_rw"][0] - np.log(np.exp(fh["ln_prob_rw"]).sum())
        print(N, betaPV, N-betaPV)
        assert np.abs(1 - betaPV/N) < 1e-2

N betaPV difference
0.2829214450028523 0.28276142935893817 0.00016001564391415757
0.3127110344609224 0.3125181844159339 0.00019285004498853509
0.3456402287398188 0.34540825464818914 0.00023197409162967197
0.3820402993452214 0.3817618872672399 0.0002784120779814603
0.4222774542216826 0.421944174504675 0.0003332797170076396
0.46675648379027573 0.4663587266639026 0.00039775712637313276
0.5159247687089498 0.5154517276198478 0.0004730410891019554
0.570276678959833 0.5697164104095647 0.0005602685502683613
0.6303583950694145 0.6296979935758966 0.0006604014935178704
0.696773183838372 0.695999122080295 0.0007740617580769271
0.770187163519557 0.7692858599548528 0.0009013035647041923
0.8513355980068428 0.8502942855654231 0.001041312441419695
0.9410297679934123 0.9398377446931344 0.0011920233002779002
1.0401644817643028 1.0388148221072249 0.0013496596570778951
1.1497263127984105 1.14821809969524 0.0015082131031705082
1.2708026900299596 1.2691437797302167 0.001658910299742855
1.4045920241185472 1.4

Did this tutorial work as expected? Did you find any inconsistencies or have any comments? Please [contact](../../../CONTACT.rst) us. Any feedback is appreciated!